In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/China_Data
/Users/samantha/QuantUS-Plugins-CEUS


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [93]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/p11/new_v1/CEUS-24685.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [94]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [95]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [110]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/p11/new_v1/v1_voi.nii.gz'
seg_loader_kwargs = {}

In [111]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis

In [112]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [113]:
analysis_type = 'curves'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [114]:
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: []


In [115]:
# Set frame rate (adjust this value to match your actual video fps)
image_data.frame_rate = 1  # e.g., 30 fps

analysis_kwargs = {
    'ax_vox_ovrlp': 50,
    'sag_vox_ovrlp': 50,
    'cor_vox_ovrlp': 50,
    'ax_vox_len': 20.0,
    'sag_vox_len': 20.0,
    'cor_vox_len': 20.0,
}


In [116]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves: 100%|██████████| 181/181 [00:01<00:00, 110.43it/s]


In [117]:
import numpy as np

# Get raw image and mask
img = image_data.intensities_for_analysis
mask = seg_data.seg_mask

voxel_vol = np.prod(image_data.pixdim)
num_voxels = np.sum(mask > 0)

# Recompute TIC to match GUI
tic = []
for t in range(img.shape[3]):
    frame = img[:, :, :, t]
    tic.append(np.exp(frame[mask > 0] / 24.09).mean() * voxel_vol * num_voxels)
tic = np.array(tic)

# Baseline subtraction (same as GUI)
tic = tic - np.mean(tic[:2])
if np.any(tic < 0):
    tic = tic + np.abs(np.min(tic))
else:
    tic = tic - np.min(tic)

# Replace the TIC in the analysis object
analysis_obj.curves[0]['TIC'] = list(tic)

## Curve Quantification

In [118]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [119]:
function_names = [] # Empty list will use all functions
output_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/output/curve_quant_raw.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [120]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

In [121]:
import numpy as np
data = curve_quant.data_dict[0]
print(f"{'Parameter':<30} {'Value':>15}")
print("-" * 47)
for key, value in data.items():
    if isinstance(value, float):
        print(f"{key:<30} {value:>15.4f}")
    else:
        print(f"{key:<30} {str(value):>15}")

# Volume calculation
pixdim = image_data.pixdim
voxel_vol_mm3 = np.prod(pixdim)
n_voxels = int(np.sum(seg_data.seg_mask > 0))
vol_mm3 = n_voxels * voxel_vol_mm3

print(f"\n{'VOI Volume':<30}")
print("-" * 47)
print(f"{'Voxel spacing (mm)':<30} {str(pixdim):>15}")
print(f"{'Voxels in VOI':<30} {n_voxels:>15,}")
print(f"{'Volume (mm³)':<30} {vol_mm3:>15.1f}")
print(f"{'Volume (cm³)':<30} {vol_mm3/1000:>15.2f}")

Parameter                                Value
-----------------------------------------------
Scan Name                       CEUS-24685.nii
Segmentation Name                       v1_voi
AUC_NoFit_TIC                          33.4199
AUC_full_TIC                   3405776799.3345
PE_full_TIC                      83962599.8813
TP_full_TIC                             8.1279
MTT_full_TIC                           48.9564
T0_full_TIC                            14.0128
Mu_full_TIC                             3.2924
Sigma_full_TIC                          1.0941
PE_Ix_full_TIC                              22
WashIn_Mean_TIC                  19230259.7788
WashIn_Std_TIC                   30324635.4367
WashIn_Max_TIC                   85336833.7337
WashIn_Min_TIC                          0.0000
WashIn_Median_TIC                  154271.8129
WashIn_Variance_TIC            919583514367160.1250
WashIn_Skewness_TIC                     1.2810
WashIn_Kurtosis_TIC                    -0.0107
WashIn_

## Save Results to CSV

In [122]:
import pandas as pd
import os
from datetime import datetime

out_path = "/Users/samantha/Desktop/ultrasound lab stuff/china data/p11/new_v1/paramap/curve_quant.csv"

data = curve_quant.data_dict[0]
df = pd.DataFrame([data])
df["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

scan_col = "Scan Name"

if os.path.exists(out_path):
    existing = pd.read_csv(out_path)
    if "Timestamp" not in existing.columns:
        existing["Timestamp"] = pd.NaT
        existing.to_csv(out_path, index=False)
    df.to_csv(out_path, mode="a", header=False, index=False)
    print(f"Appended to {out_path}")
else:
    df.to_csv(out_path, index=False)
    print(f"Created {out_path}")

Appended to /Users/samantha/Desktop/ultrasound lab stuff/china data/p11/new_v1/paramap/curve_quant.csv
